# 02 - Feature engineering, model training and experiment tracking
**Tasks 2-4.** Runs the exact training entry point CI uses (`cardiorisk.train.run_training`).
Three model families (Logistic Regression, Random Forest, RBF-SVM) are tuned with GridSearchCV
(stratified 5-fold CV on the training split). Every candidate is logged to MLflow.
The winner is picked by **mean CV ROC-AUC** - the test set is only used for the final report.

After this runs, open the MLflow UI from a terminal in the repo root:
`mlflow ui --backend-store-uri sqlite:///mlflow.db` then http://127.0.0.1:5000

In [ ]:
# Make the repo importable whether Jupyter was started in the repo root or in notebooks/
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "cardiorisk").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Repo root:", ROOT)

In [ ]:
%matplotlib inline
import logging
import pandas as pd
from IPython.display import Image, display
from cardiorisk import config as C
from cardiorisk.features import build_preprocessor
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
pd.set_option("display.width", 200, "display.max_columns", 30)

## 1. The preprocessing pipeline (shared by all models)

In [ ]:
build_preprocessor()

## 2. Tune, compare, track, gate and package
Takes about 1 minute on a laptop.

In [ ]:
from cardiorisk.train import run_training
result = run_training()
print("Selected model:", result["selected_model"])

## 3. Comparison table (CV = mean of 5 folds on the training split; test = held-out 20%)

In [ ]:
table = result["comparison"]
cols = ["model"] + [c for c in table.columns if c.endswith("_mean")] + [c for c in table.columns if c.startswith("test_")]
table[cols]

In [ ]:
table[["model", "best_params"]]

## 4. Plots (also logged to MLflow under each run's `plots/` folder)

In [ ]:
for name in ["model_cv_comparison", "all_models_roc"]:
    display(Image(filename=str(C.FIGURES_DIR / f"{name}.png")))
best = result["selected_model"]
for suffix in ["confusion_matrix", "permutation_importance"]:
    display(Image(filename=str(C.FIGURES_DIR / f"{best}_{suffix}.png")))

## 5. Packaged model metadata (what the API reads)

In [ ]:
import json
print(json.dumps(result["metadata"], indent=2))